# Notebook 02 - Baseline Model Evaluation

This notebook implements baseline machine learning models for DDoS detection using the reduced feature set obtained from exploratory data analysis. 

The objectives of this notebook are:

- Load and preprocess the CICIDS2017 dataset
- Reuse the selected feature set from Notebook 01
- Train baseline models (Logistic Regression and Random Forest)
- Evaluate performance using standard metrics
- Generate ROC curves
- Simulate real-time behaviour using a time-based split
- Analysis feature importance and check for potential leakage

# 1. Import Required Libraries

This section imports the core libraries required for data handling, modelling, evaluation, and visualisation. 

In [ ]:
# Import core libraries
import numpy as np
import pandas as pd
import json

# Import ML libraries
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve

# Import baseline models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Plotting libraries
import matplotlib.pyplot as plt
from pathlib import Path

# 2. Load and Preprocess Dataset

The CICIDS2017 Friday Afternoon DDoS dataset is loaded and cleaned.

Preprocessing steps:
- Remove leading/trailing whitespace from column names
- Replace infinite values with NaN
- Remove rows containing missing values

In [ ]:
# Load dataset
df = pd.read_csv("../data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")

# Clean column names
df.columns = df.columns.str.strip()

# Handle missing and infinite values
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.dropna().reset_index(drop=True)

# Print summary of the dataset after cleaning
print("Remaining missing values: ", df.isnull().sum().sum())
df.head()

# 3. Load Selected Feature Set

The reduced features list generated during exploratory data analysis (Notebook 01) is loaded.

Only numeric features that exist in the current dataset are retained to ensure consistency. 

In [ ]:
# Load committed feature list from EDA
features_path  = Path("../artifacts/selected_features.json")
if not features_path.exists():
    raise FileNotFoundError(f"Selected features file not found at {features_path.resolve()}")

# Load selected features
with features_path.open("r", encoding="utf-8") as f:
    selected_features = json.load(f)

# Ensure features exist and are numeric
numeric_cols = set(df.select_dtypes(include=[np.number]).columns)
selected_features = [c for c in selected_features if c in numeric_cols]

# Check if we have any valid features left after filtering
print(f"Loaded {len(selected_features)} numeric features.")
selected_features

# 4. Identify Label Column and Prepare Feature Matrix

The label column is identified dynamically to avoid hardcoding.

The feature matrix (X) and label vector (y) are constructed using the selected feature subset.

In [ ]:
# Identify label column
label_candidates = [c for c in df.columns if c.lower() == "label"]
if not label_candidates:
    raise ValueError("No label column found in the dataset.")

label_column = label_candidates[0]

# Print label distribution to check for class imbalance
print("Label column: ", label_column)
print(df[label_column].value_counts())

# Feature matrix and labels
X = df[selected_features].copy()
y = df[label_column].copy()

# Print shapes of features and labels
print("X shape: ", X.shape)

# 5. Encode Target Labels

Labels are encoded numerically to allow compatibility with scikit-learn models.

Binary classification is assumed (Benign vs DDoS).

In [ ]:
# Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# Print encoded labels and their corresponding classes
print("Classes: ", list(label_encoder.classes_))

# 6. Baseline Train/Test Split

The dataset is split into training and testing sets using stratification to preserve class distribution.

This split represents standard offline evaluation. 

In [ ]:
# Train/test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Print shapes of train and test sets
print("Train set shape: ", X_train.shape)
print("Test set shape: ", X_test.shape)

# 7. Feature Standardisation

StandardScaler is applied to the training set and then used to transform the test set.

Scaling is required for Logistic Regression but not for Random Forest.

In [ ]:
# Standardize features for linear models
scaler = StandardScaler()

# Scale train and test sets
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 8. Logistic Regression Baseline Model

A Logistic Regression model is trained as a linear baseline classifier.

Performance is evaluated using:
- Classification report
- Confusion matrix

In [ ]:
# Logistic Regression baseline
logreg = LogisticRegression(max_iter=1000)
logreg.fit(X_train_scaled, y_train)

# Evaluate Logistic Regression
y_pred_logreg = logreg.predict(X_test_scaled)

# Print classification report and confusion matrix for Logistic Regression
print("Logistic Regression Classification Report: ")
print(classification_report(y_test, y_pred_logreg, target_names=label_encoder.classes_))
print(confusion_matrix(y_test, y_pred_logreg))

# 9. Random Forest Baseline Model

A Random Forest classifier is trained as a non-linear ensemble baseline.

This model often performs well on tabular network traffic data.

In [ ]:
# Random Forest baseline
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Evaluate Random Forest
y_pred_rf = rf.predict(X_test)

# Print classification report and confusion matrix for Random Forest
print("Random Forest Report: ")
print(classification_report(y_test, y_pred_rf, target_names=label_encoder.classes_))
print(confusion_matrix(y_test, y_pred_rf))

# 10. ROC Curve and AUC Evaluation

Receiver Operating Characteristic (ROC) curves are generated using predicted probabilities.

This evaluates the model's ability to discriminate between classes across varying classification thresholds.

In [ ]:
# ROC Curves
y_prob_logreg = logreg.predict_proba(X_test_scaled)[:, 1]
y_prob_rf = rf.predict_proba(X_test)[:, 1]

# Calculate false positive rates and true positive rates for both models
fpr_logreg, tpr_logreg, _ = roc_curve(y_test, y_prob_logreg)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)

# Plot ROC curves for both models on the same graph
# TODO: Adjusting the color of the dashed line for better visibility
plt.figure(figsize=(6, 5))
plt.plot(fpr_logreg, tpr_logreg, label="Logistic Regression")
plt.plot(fpr_rf, tpr_rf, label="Random Forest")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
plt.show()

# 11. Simulated Real-Time Evaluation (Time-Based Split)

To approximate real-time deployment conditions, a time-based split is performed:

- The first 80% of flows are used for training
- The remaining 20% are treated as future traffic

This simulates sequential inference without shuffling.

In [ ]:
# Time-based split (no shuffle)
split_idx = int(len(X) * 0.8)

X_train_time, X_test_time = X.iloc[:split_idx], X.iloc[split_idx:]
y_train_time, y_test_time = y_encoded[:split_idx], y_encoded[split_idx:]

# Print shapes of time-based splits to verify correct splitting
print("Time split sizes: ", X_train_time.shape, X_test_time.shape)

# 12. Logistic Regression (Time-Based Evaluation)

Logistic Regression is retrained using the time-ordered split to simulate deployment on future traffic.

In [ ]:
# Logistic Regression (time split)
scaler_time = StandardScaler()
X_train_time_scaled = scaler_time.fit_transform(X_train_time)
X_test_time_scaled = scaler_time.transform(X_test_time)

# Train Logistic Regression on time-based split
log_reg_time = LogisticRegression(max_iter=1000)
log_reg_time.fit(X_train_time_scaled, y_train_time)

# Evaluate 
y_pred_logreg_time = log_reg_time.predict(X_test_time_scaled)
print("Logistic Regression (Time Split) Classification Report: ")
print(classification_report(y_test_time, y_pred_logreg_time, target_names=label_encoder.classes_))

# 13. Random Forest (Time-Based Evaluation)

Random Forest is evaluated using the same time-based split to assess performance under simulated real-time conditions. 

In [ ]:
# Random Forest (time split)
rf_time = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_time.fit(X_train_time, y_train_time)

# Evaluate
y_pred_rf_time = rf_time.predict(X_test_time)

# Print classification report for Random Forest on time-based split
print("Random Forest (Time Split) Classification Report: ")
print(classification_report(y_test_time, y_pred_rf_time, target_names=label_encoder.classes_))

# 14. Feature Importance Analysis

Random Forest feature importance values are examined to identify which features contribute most strongly to classification decisions.

Feature importance scores are derived from the Random Forest model trained on the standard stratified train/test split.

In [ ]:
# Random Forest feature importance
importances = pd.Series(rf.feature_importances_, index=selected_features)
importances.sort_values(ascending=False).head(15)

# 15. Single-Feature AUC Leakage Check

Each feature is individually evaluated using AUC to ensure that no single feature perfectly separates classes.

This helps identify potential data leakage or unrealistic separability.

In [ ]:
# Single-feature AUC leakage check
single_feature_auc = []
for col in selected_features:

    try:
        auc = roc_auc_score(y_encoded, df[col].values)
        single_feature_auc.append((col, auc))
        
    except ValueError:
        continue

# Sort features by how much their AUC deviates from 0.5 (random guessing)
single_feature_auc.sort(key=lambda x: abs(x[1] - 0.5), reverse=True)
single_feature_auc[:10]